# 1007 25일차

## 1. RAG

질문으로 찾은 청크를 질문과 함께 모델에 넣어 답을 받는 방식 (Retrieval-Augmented Generation)

모델만 쓰면 학습한 지식으로만 답함. 문서 없이 "삼성전자의 창업자는 누구인가요?"를 물으면 이병철을 "이건희의 할아버지"라고 답함 (실제로는 아버지)

1. 검색 : `retriever.invoke(질문)`으로 가까운 청크 `k`개를 찾음 (day24 RAG §5)
2. 붙이기 : 찾은 청크와 질문을 프롬프트 하나로 만듦
3. 생성 : 프롬프트를 `model.invoke`에 넣어 답을 받음

- 직접 붙일 때는 찾은 2개 중 첫 번째 청크만 f-string으로 질문 앞에 넣음
- 청크와 질문 사이에 "위 내용에 근거해 다음 질문에 답변하세요"를 넣음

#### cf) 문서에 없는 질문

txt 3개에 창업자 내용이 없어도 retriever는 가장 가까운 청크 `k`개를 돌려줌 (day24 RAG §5)

- "삼성전자 사업 전망" 청크를 붙이고 창업자를 물으면, "위 내용에 근거해"라고 해도 청크에 없는 이병철·이건희 이야기를 학습한 지식으로 답함
- 그래서 §2 프롬프트에는 정보가 없으면 "주어진 정보로는 답변할 수 없습니다."라고 답하라는 지시를 넣음

## 2. RAG 체인

검색 → 붙이기 → 생성을 함수 두 개로 묶은 체인

단계마다 따로 부르지 않고 `invoke` 한 번으로 질문부터 답까지 받음

![rag_chain.invoke에 질문을 넣으면 ① retriever가 질문을 임베딩해 Chroma12 청크 52개 중 가까운 2개를 context에 담고, ② docu_chain이 두 청크를 이어 붙여 프롬프트에 채운 뒤 모델과 StrOutputParser를 거쳐 answer를 만든다. dict에는 input, context, answer 순으로 키가 쌓인다](assets/rag-chain-flow.svg)

1. `create_stuff_documents_chain(model, prompt)` : 찾은 문서를 전부 `{context}`에 채워 `prompt | model | StrOutputParser`로 넘김 (day21 RAG §3, §4)
2. `create_retrieval_chain(retriever, docu_chain)` : `{"input": 질문}`으로 검색한 문서를 `context`에, 1번 체인의 답을 `answer`에 담음

- stuff는 찾은 문서를 프롬프트 하나에 전부 채운다는 뜻. 문서 사이는 `\n\n`으로 이음
- 프롬프트 변수 이름은 `{context}`·`{input}`으로 고정. 체인 함수가 이 이름으로 값을 넘겨서, `{context}`가 없으면 체인을 만들 때 에러
- `ChatPromptTemplate.from_template` : 템플릿 전체가 사람 메시지(`HumanMessage`) 하나로 들어가는 채팅용 프롬프트. `PromptTemplate`(day21 RAG §2)은 채우면 문자열

| | 직접 붙이기 (§1) | 체인 |
|---|---|---|
| 넣는 문서 | 첫 번째 청크 1개 | 찾은 `k`개 전부 |
| 프롬프트 | f-string | `ChatPromptTemplate`의 `{context}`·`{input}` |
| 호출 | `retriever.invoke` → `model.invoke` | `rag_chain.invoke({"input": 질문})` 한 번 |
| 결과 | `AIMessage` | dict (§2-1) |

### 2-1. 체인 결과

`rag_chain.invoke()`는 dict를 돌려줌. `response.keys()` → `dict_keys(['input', 'context', 'answer'])`

| 키 | 내용 |
|---|---|
| `input` | 넣은 질문 |
| `context` | 검색한 `Document` 리스트 |
| `answer` | 모델 답 (문자열) |

- 답만 보려면 `response['answer']`
- `context`에는 검색한 `Document`가 그대로 들어 있음. 이어 붙인 문자열은 프롬프트에만 들어감

#### cf) `langchain_classic`

langchain 1.x에는 `chains`가 없어서 `create_stuff_documents_chain`·`create_retrieval_chain`은 `langchain_classic`에서 가져옴

- `from langchain.chains import ...`는 1.x에서 `ModuleNotFoundError`

## 3. Gradio

파이썬 함수를 웹 화면으로 띄우는 라이브러리

§2 체인을 부르는 함수를 `gr.ChatInterface`에 넘기면 브라우저에서 질문하고 답을 받는 챗봇이 됨

1. `gr.ChatInterface(fn, title)` : 채팅 화면을 만듦. `title`은 화면 위 제목이자 브라우저 탭 이름
2. `fn(message, history)` : 메시지를 보낼 때마다 불리는 함수. 입력한 메시지와 이전 대화 리스트를 받아 답 문자열을 돌려줌
3. `launch()` : 로컬 웹 서버를 띄움. 터미널에 나오는 주소(`http://127.0.0.1:7860`)를 브라우저로 엶

- `fn`은 `rag_chain.invoke` 결과에서 `answer`만 돌려줌 (§2-1)
- `history`를 체인에 넘기지 않아서 이전 대화는 프롬프트에 안 들어가고 질문마다 따로 검색함

## 4. FAISS

벡터 검색 라이브러리 FAISS를 LangChain 벡터 저장소로 감싼 것

FAISS는 대규모 검색 속도가 장점인 라이브러리 (day23 RAG §1-1). Chroma와 같은 `VectorStore`라 `similarity_search`·`as_retriever`는 똑같이 씀

| | 저장 | 불러오기 |
|---|---|---|
| 코드 | `FAISS.from_documents()` → `save_local()` | `FAISS.load_local()` |
| 하는 일 | 청크를 임베딩해 메모리에 저장소를 만들고 파일로 씀 | 파일을 읽어 메모리에 올림 |
| 임베딩 인자 | `embedding` | `embeddings` |

1. `folder_path` : 저장 폴더
2. `index_name` : 파일 이름. `이름.faiss`(벡터)와 `이름.pkl`(문서·id 연결) 두 파일로 저장
3. `allow_dangerous_deserialization=True` : `.pkl`을 pickle로 읽어서 직접 허용해야 열림. 조작된 pickle은 여는 순간 코드가 실행될 수 있어 기본값은 `False`

- Chroma(day24 RAG §3)는 `from_documents`에서 바로 폴더에 쓰이지만, FAISS는 `save_local`을 불러야 파일로 남음
- 저장을 다시 실행하면 저장소를 새로 만들어 파일을 덮어씀. 18개 그대로 (Chroma는 쌓임, day24 RAG §3 cf)
- 불러올 때 임베딩 모델·`dimensions`가 저장할 때와 같아야 함. Faiss17은 `dimensions=5`로 저장해서 5차원
- §3 챗봇의 Chroma를 FAISS로 바꿔도 retriever를 쓰는 법이 같아서 §2 체인과 화면 코드는 그대로

### 4-1. 저장 구조

벡터는 FAISS 인덱스에, 청크 원문은 문서 저장소에 따로 담고 행 번호로 둘을 이음

| 속성 | 내용 |
|---|---|
| `index` | 벡터. 들어온 순서대로 행 번호 0, 1, 2, … |
| `docstore` | 청크 원문·메타데이터. `docstore._dict`가 id → `Document` dict |
| `index_to_docstore_id` | 행 번호 → docstore id |

1. `index`에서 질문 벡터와 가까운 행 번호를 찾음
2. `index_to_docstore_id`로 행 번호를 id로 바꿈
3. `docstore`에서 그 id의 `Document`를 꺼내 돌려줌

- `index.d` : 벡터 차원 수, `index.ntotal` : 들어 있는 벡터 수 (Chroma의 `_collection.count()`)
- Faiss17은 `d` 5, `ntotal` 18. samsung 청크 9개 다음에 nvidia 청크 9개가 0~17번

#### cf) 인덱스 직접 만들기

`from_documents` 없이 빈 벡터 저장소를 직접 만드는 방법. 잘 쓰지 않음

1. `faiss.IndexFlatL2(차원)` : 넣은 벡터 전부와 L2(유클리드) 거리를 재는 인덱스
2. `InMemoryDocstore()` : 빈 문서 저장소
3. `index_to_docstore_id={}` : 빈 연결표

- 셋을 `FAISS(...)`에 넣으면 문서가 0개인 벡터 저장소 (`index.ntotal` 0)
- 차원은 임베딩 차원과 같아야 해서 `embed_query`로 문장 하나를 임베딩한 길이로 정함. `from_documents`는 첫 임베딩의 길이로 인덱스를 만듦